In [1]:
# ==============================================================================
# SCRIPT:         colab_session_context_generator_v1.6.ipynb (Synchronized Output)
# OPERATION:      Session Context Layer
# PURPOSE:        Preserves the original row count of the source sheet, ensuring the
#                 final output is perfectly synchronized. Calculations are skipped for
#                 rows with missing data, leaving new feature columns blank (NaN).
# ==============================================================================

# --- 1. Setup ---
!pip install -q gspread
import pandas as pd
from google.colab import auth, drive
import gspread
from google.auth import default
import os
import numpy as np # Import numpy for NaN handling

print("Authenticating...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive.mount('/content/drive', force_remount=True)
print("✅ Setup complete.")

# --- 2. CONFIGURATION ---
SPREADSHEET_NAME = 'RAW_Requests'
WORKSHEET_NAME = 'raw_requests_VVSpolished'
OUTPUT_FOLDER_PATH = '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/'
OUTPUT_FILENAME = 'RAW_Requests_v4_session_context_FINAL.csv'
SESSION_COL = 'Session_ID'
TIMESTAMP_COL = 'timestamp'

def make_headers_unique(header_list):
    counts, unique_headers = {}, []
    for i, header in enumerate(header_list):
        header = str(header).strip() if str(header).strip() != "" else f"blank_{i}"
        if header not in counts:
            counts[header] = 1; unique_headers.append(header)
        else:
            counts[header] += 1; unique_headers.append(f"{header}_{counts[header]}")
    return unique_headers

try:
    # --- 3. Load Full Dataset ---
    print(f"\nLoading full dataset from '{WORKSHEET_NAME}'...")
    worksheet = gc.open(SPREADSHEET_NAME).worksheet(WORKSHEET_NAME)
    all_values = worksheet.get_all_values()
    unique_headers = make_headers_unique(all_values[0])
    df_full = pd.DataFrame(all_values[1:], columns=unique_headers)
    initial_row_count = len(df_full)
    print(f"  > Loaded {initial_row_count} total rows. This count will be preserved.")

    # --- 4. Create Sanitized DataFrame for Calculation ---
    print("Preparing a sanitized subset of data for calculations...")
    df_calc = df_full.copy()
    df_calc.replace('', np.nan, inplace=True)
    df_calc.dropna(subset=[SESSION_COL, TIMESTAMP_COL], inplace=True)

    df_calc['dt_timestamp'] = pd.to_datetime(df_calc[TIMESTAMP_COL], format='%Y:%m:%d %H:%M:%S', errors='coerce')
    df_calc.dropna(subset=['dt_timestamp'], inplace=True)
    print(f"  > Created temporary calculation dataset with {len(df_calc)} valid rows.")

    # --- 5. Perform Calculations ONLY on the Clean Data ---
    print("Calculating session features on sanitized data...")
    session_stats = df_calc.groupby(SESSION_COL)['dt_timestamp'].agg(
        session_start_time='min',
        session_end_time='max'
    ).reset_index()
    session_stats['session_duration_sec'] = (session_stats['session_end_time'] - session_stats['session_start_time']).dt.total_seconds()

    # --- 6. MERGE Features back onto the FULL, Original DataFrame ---
    print("\nMerging calculated features back onto the original, full dataset...")
    df_enriched = pd.merge(df_full, session_stats, on=SESSION_COL, how='left')

    # Create a temporary datetime column on the full set for calculations
    df_enriched['dt_timestamp'] = pd.to_datetime(df_enriched[TIMESTAMP_COL], format='%Y:%m:%d %H:%M:%S', errors='coerce')

    # --- 7. FINAL FEATURE CALCULATION (will produce NaNs where data is missing) ---
    print("Calculating final features for all rows...")
    time_elapsed_sec = (df_enriched['dt_timestamp'] - df_enriched['session_start_time']).dt.total_seconds()

    df_enriched['time_in_session_sec'] = time_elapsed_sec.round(0) # Keep as float to allow NaNs
    df_enriched['session_progress_ratio'] = (time_elapsed_sec / df_enriched['session_duration_sec']).round(6)

    # Cleanup and define final output columns
    final_cols = list(df_full.columns) + ['session_duration_sec', 'time_in_session_sec', 'session_progress_ratio']
    df_final = df_enriched[final_cols]

    final_row_count = len(df_final)

    # --- 8. Save ---
    full_output_path = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_FILENAME)
    df_final.to_csv(full_output_path, index=False)

    print(f"\n✅ Mission Success. Final data saved to:\n   > '{full_output_path}'")

    # --- FINAL VERIFICATION ---
    print("\n--- Final Verification Checks ---")
    if initial_row_count == final_row_count:
        print(f"  > PASSED: Row count is synchronized. Initial: {initial_row_count}, Final: {final_row_count}.")
    else:
        print(f"  > FAILED: Row count MISMATCH. Initial: {initial_row_count}, Final: {final_row_count}.")

    nan_rows = df_final[df_final['session_duration_sec'].isnull()]
    print(f"  > INFO: Found {len(nan_rows)} rows with blank features due to missing initial data.")

except Exception as e:
    print(f"❌ An unrecoverable error occurred: {e}")

Authenticating...
Mounted at /content/drive
✅ Setup complete.

Loading full dataset from 'raw_requests_VVSpolished'...
  > Loaded 4776 total rows. This count will be preserved.
Preparing a sanitized subset of data for calculations...
  > Created temporary calculation dataset with 4766 valid rows.
Calculating session features on sanitized data...

Merging calculated features back onto the original, full dataset...
Calculating final features for all rows...
❌ An unrecoverable error occurred: 'session_duration_sec'


/tmp/ipython-input-1166442479.py:56: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_calc.replace('', np.nan, inplace=True)
